In [1]:
import subprocess, os, time
env = {**os.environ, "QT_QPA_PLATFORM": "offscreen"}
turtlesim = subprocess.Popen(["ros2", "run", "turtlesim", "turtlesim_node"], env=env)
time.sleep(2)

QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-jovyan'
[WARN] [1791363098.499582234] [rcl]: ROS_LOCALHOST_ONLY is deprecated but still honored if it is enabled. Use ROS_AUTOMATIC_DISCOVERY_RANGE and ROS_STATIC_PEERS instead.
[WARN] [1791363098.499623271] [rcl]: 'localhost_only' is disabled, 'automatic_discovery_range' and 'static_peers' will be used.
[INFO] [1791363098.535241024] [turtlesim]: Starting turtlesim with node name /turtlesim
[INFO] [1791363098.605734884] [turtlesim]: Spawning turtle [turtle1] at x=[5.544445], y=[5.544445], theta=[0.000000]


In [2]:
# pip install ipycanvas   (if not already in the image)
import math, time, threading
import rclpy
from rclpy.node import Node
from turtlesim.msg import Pose
from geometry_msgs.msg import Twist
from ipycanvas import Canvas, hold_canvas
import ipywidgets as w
from IPython.display import display

S = 40  # px per turtlesim unit (11x11 world -> 440x440)

class TurtleView(Node):
    def __init__(self):
        super().__init__("jupyter_turtle_view")
        self.canvas = Canvas(width=11*S, height=11*S)
        self.label = w.Label()
        self.trail, self.last_draw = [], 0.0
        self.pub = self.create_publisher(Twist, "/turtle1/cmd_vel", 10)
        self.create_subscription(Pose, "/turtle1/pose", self.on_pose, 10)

    def on_pose(self, p):
        self.trail.append((p.x*S, (11-p.y)*S))
        self.trail = self.trail[-2000:]
        now = time.time()
        if now - self.last_draw < 0.05:      # ~20 FPS cap
            return
        self.last_draw = now
        c, (x, y) = self.canvas, self.trail[-1]
        with hold_canvas(c):
            c.fill_style = "#4556ff"; c.fill_rect(0, 0, 11*S, 11*S)
            c.stroke_style = "white"; c.line_width = 2
            c.begin_path(); c.move_to(*self.trail[0])
            for pt in self.trail[1:]: c.line_to(*pt)
            c.stroke()
            a = -p.theta                       # flip y axis
            c.fill_style = "#2e9e4f"; c.begin_path()
            c.move_to(x + 16*math.cos(a), y + 16*math.sin(a))
            c.line_to(x + 11*math.cos(a+2.5), y + 11*math.sin(a+2.5))
            c.line_to(x + 11*math.cos(a-2.5), y + 11*math.sin(a-2.5))
            c.close_path(); c.fill()
        self.label.value = f"x={p.x:.2f}  y={p.y:.2f}  θ={p.theta:.2f}"

    def drive(self, lin, ang):
        m = Twist(); m.linear.x, m.angular.z = lin, ang
        self.pub.publish(m)

rclpy.init()
view = TurtleView()
threading.Thread(target=rclpy.spin, args=(view,), daemon=True).start()

def btn(txt, lin, ang):
    b = w.Button(description=txt, layout=w.Layout(width="60px"))
    b.on_click(lambda _: view.drive(lin, ang)); return b

pad = w.HBox([btn("←", 0, 1.5), btn("↑", 1.5, 0), btn("↓", -1.5, 0), btn("→", 0, -1.5)])
display(w.VBox([view.canvas, view.label, pad]))

ModuleNotFoundError: No module named 'ipycanvas'

In [3]:
%pip install ipycanvas

ERROR: Could not install packages due to an OSError: [Errno 13] Permission denied: '/opt/venv/lib/python3.12/site-packages/ipycanvas'
Check the permissions.


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import math, time, threading
import rclpy
from rclpy.node import Node
from turtlesim.msg import Pose
from geometry_msgs.msg import Twist
import ipywidgets as w
from IPython.display import display

S = 40  # px per turtlesim unit (11x11 world -> 440x440)
W = 11 * S

class TurtleView(Node):
    def __init__(self):
        super().__init__("jupyter_turtle_view")
        self.screen = w.HTML()
        self.label = w.Label()
        self.trail, self.last_draw = [], 0.0
        self.pub = self.create_publisher(Twist, "/turtle1/cmd_vel", 10)
        self.create_subscription(Pose, "/turtle1/pose", self.on_pose, 10)
        self.render(5.5, 5.5, 0.0)

    def render(self, x, y, theta):
        px, py = x * S, (11 - y) * S
        pts = " ".join(f"{a:.1f},{b:.1f}" for a, b in self.trail)
        self.screen.value = (
            f'<svg width="{W}" height="{W}" style="background:#4556ff">'
            f'<polyline points="{pts}" fill="none" stroke="white" stroke-width="2"/>'
            f'<polygon points="16,0 -9,-7 -9,7" fill="#2e9e4f" '
            f'transform="translate({px:.1f},{py:.1f}) rotate({-math.degrees(theta):.1f})"/>'
            f'</svg>')
        self.label.value = f"x={x:.2f}  y={y:.2f}  θ={theta:.2f}"

    def on_pose(self, p):
        self.trail.append((p.x * S, (11 - p.y) * S))
        self.trail = self.trail[-2000:]
        now = time.time()
        if now - self.last_draw < 0.05:   # ~20 FPS cap
            return
        self.last_draw = now
        self.render(p.x, p.y, p.theta)

    def drive(self, lin, ang):
        m = Twist(); m.linear.x, m.angular.z = float(lin), float(ang)
        self.pub.publish(m)

# re-run safe
try:
    view.destroy_node()
except NameError:
    pass
if not rclpy.ok():
    rclpy.init()

view = TurtleView()
threading.Thread(target=rclpy.spin, args=(view,), daemon=True).start()

def btn(txt, lin, ang):
    b = w.Button(description=txt, layout=w.Layout(width="60px"))
    b.on_click(lambda _: view.drive(lin, ang)); return b

pad = w.HBox([btn("←", 0, 1.5), btn("↑", 1.5, 0), btn("↓", -1.5, 0), btn("→", 0, -1.5)])
display(w.VBox([view.screen, view.label, pad]))

[WARN] [1791363166.032665343] [rcl]: ROS_LOCALHOST_ONLY is deprecated but still honored if it is enabled. Use ROS_AUTOMATIC_DISCOVERY_RANGE and ROS_STATIC_PEERS instead.
[WARN] [1791363166.032690109] [rcl]: 'localhost_only' is disabled, 'automatic_discovery_range' and 'static_peers' will be used.
